# Complexity-08e — Le compte à rebours : des machines finies qui attendent une contradiction (NF, C₀)

*Greffe D de [#27537](https://github.com/jsboige/CoursIA/issues/27537) (EPIC [#20607](https://github.com/jsboige/CoursIA/issues/20607)), voisine de `Complexity-08d` (Greffe A, PR #27483) et de [Complexity-07b — Le tour des grands nombres](Complexity-07b-LeTourDesGrandsNombres-Python.ipynb), où le castor affairé est introduit.*

**Navigation** : [<< Complexity-08d — Indépendance prouvée](Complexity-08d-IndependanceProuvee-BusyBeaver-Covering-Python.ipynb) | [Index de la série](README.md)
**Navigation** : [→ Explorer la version Rust](Complexity-08e-NF-MachinesCompteARebours-Rust.ipynb)
**Public : Recherche.** Le carnet ne suppose que les carnets précédents de la série : les machines de Turing pas à pas (01), la notion de réduction (02), le castor affairé et sa croissance (07b).

> **Statut épistémique.** Tout ce qui suit distingue trois natures de résultats : un **théorème Lean formalisé et audité**, un **calcul reproductible vérifié par simulateur indépendant**, et une **borne conditionnelle**. Les artefacts du dépôt `wlad-svennik/notes` sont **cités et épinglés par révision** ; aucun contenu n'est recopié dans ce dépôt (aucune licence détectée dans le dépôt source au 20/11/2026, demande de clarification ouverte : `wlad-svennik/notes#1`). La section de mécanique exécute **notre propre** machine-jouet, déclarée comme telle : elle reproduit le *câblage* du compte à rebours, jamais les machines réelles.

## 0. Trois statuts pour une même famille d'objets

Le dépôt source publie une famille de machines à nombre d'états décroissant (210, 201, 192, 185, 184) et, en parallèle, des machines calibrées sur une autre théorie (198 états pour C₀). Avant de citer un seul chiffre, la règle du carnet 08d s'applique ici mot pour mot : **classer l'objet avant de l'affirmer**.

1. **Théorème Lean formalisé et audité.** Pour `nf-bb185`, l'équivalence est un théorème non borné, `NFMachine185.halts_iff_nf_refutation`, accompagné d'un audit d'axiomes enregistré : trois axiomes standard de Lean et onze vérifications finies par évaluation native. Pour `nf-bb192`, l'équivalence a été vérifiée par Lean 4.35.0 sur la table littérale. C'est le statut le plus fort de la famille.
2. **Calcul reproductible.** Pour `nf-bb184`, le dépôt publie la table de la machine (empreinte SHA-256 annoncée `7764cc84…8798`), un simulateur indépendant (142 608 essais, 88 555 587 pas binaires), la reconstruction des 2 809 instructions émettrices d'axiomes, un point de contrôle de vérification de 1,84 Mo et un `SHA256SUMS`. Ce n'est pas un théorème *sur* NF : c'est la vérification finie d'une table.
3. **Borne conditionnelle.** Pour `c0-198`, le dépôt relie des machines de 198 états à l'incohérence de C₀ et énonce des bornes de la forme Con(NF) ⇒ non-arrêt ⇒ Con(NFU + Infinity). Une borne conditionnelle se cite **avec sa condition**, jamais comme un fait nu.

Le reste du carnet suit cet ordre : le mécanisme d'abord (§2), les artefacts ensuite (§3), la chaîne de compression comme méthodologie (§3), puis les ponts (§4).

## 1. NF, NFU, C₀ : le décor

La théorie **NF** (New Foundations, Quine, 1937) est une théorie des ensembles à compréhension **stratifiée** : l'ensemble {x | φ(x)} existe lorsque la formule φ respecte une stratification des variables par niveaux. Cette restriction est plus faible que la compréhension non restreinte de la théorie naïve, et plus étrange que celle de ZF : NF admet un ensemble universel et la négation de l'axiome du choix est un théorème. Sa cohérence est un problème **ouvert** depuis près de quatre-vingt-dix ans.

**NFU** est la variante avec ur-éléments (Jensen) ; l'adjonction d'axiomes d'infini et de choix y change la donne. **C₀** est une variante de la même famille, dont le dépôt source calibre des machines sur l'incohérence.

L'idée du **compte à rebours** est alors mécanique. Une machine de Turing à nombre d'états fini peut encoder la recherche de preuves d'une théorie T : elle énumère les dérivations et **s'arrête si et seulement si T dérive une contradiction**. Si l'équivalence « la machine s'arrête ⇔ T est incohérente » est elle-même un théorème — formalisé et audité — alors observer que la machine **ne s'arrête pas** établit Con(T). Autrement dit, le nombre d'états d'une machine devient une unité de mesure de la force d'une théorie : au-delà d'un seuil, la fonction du castor affairé dépasse toute fonction que T prouve totale.

C'est exactement le pont avec [07b](Complexity-07b-LeTourDesGrandsNombres-Python.ipynb) : là-bas, le castor affairé mesurait la croissance ; ici, une **borne inférieure** sur sa valeur encode l'indépendance d'une théorie. Et c'est là qu'il faut tenir la distinction : **ce carnet ne parle pas de PA196**. L'indépendance visée ici est celle de **NF**, jamais une fusion avec Peano — le dépôt source écrit lui-même que le paquet à 184 états ne contient pas de nouveau théorème Busy Beaver interne à PA.

## 2. Le mécanisme sur un jouet déclaré

La vraie machine de 184 états encode la logique du premier ordre de NF sur une bande binaire : chaque instruction contribue à l'énumération des preuves. Reproduire cela ici serait une réimplémentation sans licence et sans preuve. À la place, nous construisons **notre propre** machine-jouet sur la logique propositionnelle : un contrôle fini qui énumère les résolutions d'une base de clauses et **s'arrête lorsque la clause vide est dérivée**. Le câblage est le même ; la théorie est minuscule, et c'est précisément ce qui rend le tout exécutable et vérifiable.

Le carnet n'implémente aucun artefact du dépôt source : il implémente l'**idée** du compte à rebours, avec un oracle indépendant (la fermeture par résolution) qui permet de contrôler la machine pas à pas.

In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from itertools import combinations
import hashlib
import random

# Le système-jouet : des clauses propositionnelles sur des atomes numérotés.
# +k désigne l'atome x_k, -k sa négation.
Clause = frozenset[int]


def est_tautologie(c: Clause) -> bool:
    """Vrai si la clause contient un littéral et son opposé."""
    return any(-l in c for l in c)


def resoudre(c1: Clause, c2: Clause, pivot: int) -> Clause:
    """Résolvante de c1 et c2 sur le pivot donné (c1 contient +pivot, c2 -pivot)."""
    return (c1 - {pivot}) | (c2 - {-pivot})


def fermeture(kb: list[Clause]) -> tuple[list[Clause], bool]:
    """Oracle indépendant : fermeture par résolution, et présence de la clause vide."""
    tape = list(dict.fromkeys(kb))
    vus = set(tape)
    if any(len(c) == 0 for c in tape):
        return tape, True
    change = True
    while change:
        change = False
        instantane = list(tape)
        for c1, c2 in combinations(instantane, 2):
            for pivot in [l for l in c1 if -l in c2]:
                r = resoudre(c1, c2, pivot)
                if not r:
                    return tape, True
                if not est_tautologie(r) and r not in vus:
                    vus.add(r)
                    tape.append(r)
                    change = True
    return tape, False


K1 = [frozenset({1}), frozenset({-1, 2}), frozenset({-2})]
K2 = [frozenset({1, 2}), frozenset({-1, 2}), frozenset({1, -2}), frozenset({-1, -2})]
K3 = [frozenset({1, 2}), frozenset({-1, 2})]

for nom, kb in (("K1", K1), ("K2", K2), ("K3", K3)):
    clauses, refute = fermeture(kb)
    print(f"{nom} : {len(clauses)} clause(s) dans la fermeture, réfutation = {refute}")

K1 : 5 clause(s) dans la fermeture, réfutation = True
K2 : 8 clause(s) dans la fermeture, réfutation = True
K3 : 3 clause(s) dans la fermeture, réfutation = False


In [2]:
# Un automate explicite rend chaque resolution observable, une transition a la fois.
@dataclass
class CompteARebours:
    """Machine à contrôle fini qui énumère les résolutions, une par pas.

    États : ENUM_I -> ENUM_J -> RESOUT -> AVANCE_J -> ... -> FIN_PASSE -> ENUM_I.
    L'état HALTE signifie : la clause vide a été dérivée (la théorie se contredit).
    L'état BOULCLE est un point fixe : la fermeture est épuisée, la machine y
    tourne indéfiniment -- c'est le cas « non-arrêt » du compte à rebours.
    """
    # La bande contient la base puis ses resolvantes nouvelles ; `vus` evite les doublons.

    base: list[Clause]
    tape: list[Clause] = field(default_factory=list)
    vus: set[Clause] = field(default_factory=set)
    etat: str = "ENUM_I"
    i: int = 0
    j: int = 0
    pivots: list[int] = field(default_factory=list)
    kp: int = 0
    ajout_passe: bool = False
    pas: int = 0
    halte: bool = False

        # Une clause vide initiale est deja une refutation : aucun pas n'est necessaire.
    def __post_init__(self) -> None:
        self.tape = list(dict.fromkeys(self.base))
        self.vus = set(self.tape)
        if any(len(c) == 0 for c in self.tape):
            self.halte = True
            self.etat = "HALTE"

    # HALTE et BOULCLE sont absorbants : le compteur de pas ne progresse plus.
    def pas_once(self) -> None:
        """Une transition du contrôle fini (un pas de machine)."""
        if self.halte or self.etat == "BOULCLE":
            return
        self.pas += 1
        # ENUM_I choisit la premiere clause de chaque paire ordonnee (i < j).
        if self.etat == "ENUM_I":
            if self.i >= len(self.tape):
                self.etat = "FIN_PASSE"
            else:
                self.j = self.i + 1
                self.etat = "ENUM_J"
        # ENUM_J balaie la seconde clause ; les pivots opposes alimentent RESOUT.
        elif self.etat == "ENUM_J":
            if self.j >= len(self.tape):
                self.i += 1
                self.etat = "ENUM_I"
            else:
                c1, c2 = self.tape[self.i], self.tape[self.j]
                self.pivots = [l for l in sorted(c1) if -l in c2]
                self.kp = 0
                self.etat = "RESOUT"
        # RESOUT produit les resolvantes une par une, y compris la clause vide.
        elif self.etat == "RESOUT":
            if self.kp >= len(self.pivots):
                self.etat = "AVANCE_J"
            else:
                pivot = self.pivots[self.kp]
                self.kp += 1
                r = resoudre(self.tape[self.i], self.tape[self.j], pivot)
                # La clause vide donne HALTE ; toute autre clause inedite prolonge la bande.
                if not r:
                    self.halte = True
                    self.etat = "HALTE"
                elif not est_tautologie(r) and r not in self.vus:
                    self.vus.add(r)
                    self.tape.append(r)
                    self.ajout_passe = True
        # AVANCE_J materialise le pas suivant sans calculer pendant le changement d'indice.
        elif self.etat == "AVANCE_J":
            self.j += 1
            self.etat = "ENUM_J"
        # Une nouvelle passe n'a lieu que si la precedente a ajoute une clause.
        elif self.etat == "FIN_PASSE":
            if self.ajout_passe:
                self.ajout_passe = False
                self.i = 0
                self.etat = "ENUM_I"
            # Sans nouveauté, le point fixe represente le non-arret du compte a rebours.
            else:
                self.etat = "BOULCLE"


# L'observateur borne l'attente ; il ne decide pas a la place de la machine.
def executer(machine: CompteARebours, max_pas: int = 200_000) -> CompteARebours:
    """Observe la machine jusqu'à arrêt, point fixe ou budget épuisé."""
    while not machine.halte and machine.etat != "BOULCLE" and machine.pas < max_pas:
        machine.pas_once()
    return machine


# K1 et K2 se refutent, K3 atteint sa fermeture sans contradiction.
for nom, kb in (("K1", K1), ("K2", K2), ("K3", K3)):
    m = executer(CompteARebours(kb))
    if m.halte:
        verdict = "HALTE (réfutation dérivée)"
    elif m.etat == "BOULCLE":
        verdict = "tourne indéfiniment (fermeture épuisée sans clause vide)"
    else:
        verdict = "budget épuisé sans verdict"
    print(f"{nom} : {verdict} après {m.pas} pas -- {len(m.tape)} clause(s) sur la bande")

K1 : HALTE (réfutation dérivée) après 27 pas -- 5 clause(s) sur la bande
K2 : HALTE (réfutation dérivée) après 87 pas -- 8 clause(s) sur la bande
K3 : tourne indéfiniment (fermeture épuisée sans clause vide) après 36 pas -- 3 clause(s) sur la bande


In [3]:
rng = random.Random(20261120)
atomes = (1, 2, 3)


def kb_aleatoire(tirage: random.Random, n_clauses: int) -> list[Clause]:
    """Une base de clauses aléatoire sur trois atomes, sans tautologie."""
    clauses: list[Clause] = []
    for _ in range(n_clauses):
        taille = tirage.choice((1, 2, 3))
        clause = frozenset(tirage.choice(atomes) * tirage.choice((1, -1)) for _ in range(taille))
        if not est_tautologie(clause) and clause not in clauses:
            clauses.append(clause)
    return clauses


# comptes : accords/desaccords machine-oracle, indecis (ni refute ni boucle prouvee)
accords = 0
desaccords: list[str] = []
indecis = 0
pas_max_refutation = 0

# campagne : 200 bases aleatoires, la machine doit refuter exactement quand l'oracle deriverait la clause vide
for essai in range(200):
    kb = kb_aleatoire(rng, rng.randint(1, 6))
    _, refute = fermeture(kb)
    m = executer(CompteARebours(kb), max_pas=50_000)
    if refute != m.halte:
        desaccords.append(f"essai {essai} : oracle={refute}, machine={m.halte}")
    elif refute and m.halte:
        accords += 1
        pas_max_refutation = max(pas_max_refutation, m.pas)
    elif not refute and m.etat == "BOULCLE":
        accords += 1
    else:
        indecis += 1

print(f"accords oracle/machine : {accords}/200")
print(f"désaccords : {len(desaccords)}")
print(f"indécis (budget épuisé avant point fixe) : {indecis}")
print(f"pas maximal observé pour une réfutation : {pas_max_refutation}")
for ligne in desaccords[:5]:
    print(" ", ligne)

accords oracle/machine : 200/200
désaccords : 0
indécis (budget épuisé avant point fixe) : 0
pas maximal observé pour une réfutation : 140


### Lecture du résultat

La campagne contrôle la propriété du compte à rebours sur 200 bases tirées avec une graine fixée : quand l'oracle trouve une réfutation, la machine **halte** (et le nombre de pas est mesuré) ; quand l'oracle n'en trouve pas, la machine atteint le point fixe `BOULCLE` sans avoir halté. Aucun désaccord n'est attendu, et la catégorie « indécis » compte les tirages où le budget s'épuise avant le point fixe — une machine réelle, elle, n'a pas de budget : elle tourne.

Deux précautions rendent cette lecture honnête. D'abord, l'accord porte sur un **toy** : trois atomes, des clauses courtes, une fermeture finie — rien de tout cela ne dit quoi que ce soit de NF. Ensuite, la machine-jouet n'est **pas** la machine réelle : elle partage avec elle le câblage (énumération de dérivations, arrêt sur la clause vide, non-arrêt sur fermeture épuisée), pas l'encodage. C'est exactement le geste de la section 0 : distinguer ce qui est prouvé, calculé, et simplement illustré.

## 3. Les artefacts réels, épinglés (aucun vendoring)

Le dépôt `wlad-svennik/notes` (dernier push lu le 20/11/2026 à 11:19Z) publie, dossier par dossier :

| Dossier | Objet | Statut | Ce que le dépôt publie |
|---|---|---|---|
| `nf-bb210` | Machine à 210 états + développement Lean 4 externe sur NF | table + déclarations + audit enregistré | table, déclarations de théorèmes, audit, instantané 05/11 |
| `nf-bb192` | Machine-compte à rebours à 192 états : s'arrête ssi la NF du premier ordre prouve une contradiction | **théorème vérifié par Lean 4.35.0** sur la table littérale | `COUNTDOWN-PROOF.md`, `NF192Countdown.lean`, preuve informelle PDF, certificats hors ligne |
| `nf-bb185` | Machine à 185 états, même équivalence | **théorème Lean non borné** `NFMachine185.halts_iff_nf_refutation` | audit d'axiomes (`theorem-axioms.txt`) : 3 axiomes standard + 11 vérifications natives ; traces de simulation, volumes dans le dépôt |
| `nf-bb184` | Machine à 184 états, NF pure de Quine | **calcul reproductible** | table `NF184.tm` (SHA-256 `7764cc84…8798`), 7 794 théorèmes signés, 142 608 essais / 88 555 587 pas binaires, 2 809 instructions émettrices, point de contrôle 1,84 Mo, `SHA256SUMS`, scripts de rejeu |
| `c0-198` | Deux machines à 198 états calibrées sur l'incohérence de C₀ | **borne conditionnelle** | rapports de preuve, points de contrôle reproductibles, rapport de réverse-interprétation (Con(NF) ⇒ non-arrêt ⇒ Con(NFU + Infinity)) |

**Ce que « épinglé » veut dire ici** : chaque affirmation renvoie à une révision du dépôt (par exemple `1297cce204` pour le paquet 184, `e0a8130149` et `bfc960b072` pour le 185, `299f86fd7c` et `a006d3abba` pour le 192, `33772f530c` et `9823a07d6b` pour les interprétations, `0a0c8fa599` pour C₀). Le carnet ne recopie ni table ni fichier ; il cite la révision et laisse la vérification au dépôt. La question de licence est ouverte (`wlad-svennik/notes#1`) : tant qu'elle ne reçoit pas de réponse, **zéro vendoring** reste la règle, et chaque greffe se limite à citation + consommation épinglée ou exécution de l'outil cité.

**La chaîne de compression documentée** est `210 → 201 → 192 → 185 → 184`. Le dépôt en expose la méthodologie : table à marqueur variable, testeur `D(x,y)=(J(Kx))y` et formule `((D(x,y)=x)→(D(x,x)=x))→y=x`, réordonnancement des huit rangées, états décodeurs partagés — et **les échecs conservés** (`WORK_NOTES.md` garde un contre-modèle à trois éléments pour `collapsed183`). Le prochain calcul ne fait pas de la physique : il mesure la compression elle-même.

In [4]:
# chaine documentee par le depot : 210 -> 201 -> 192 -> 185 -> 184 etats
chaine = [
    ("nf-bb210", 210),
    ("étape intermédiaire", 201),
    ("nf-bb192", 192),
    ("nf-bb185", 185),
    ("nf-bb184", 184),
]

print("chaîne de compression documentée par le dépôt (états) :")
precedent_nom, precedent = chaine[0]
for nom, n in chaine[1:]:
    delta = precedent - n
    print(f"  {precedent_nom:20s} {precedent:3d} -> {nom:20s} {n:3d} : -{delta} état(s), {delta / precedent:.2%} de la taille précédente")
    precedent_nom, precedent = nom, n

# reduction totale de 210 a 184
total = chaine[0][1] - chaine[-1][1]
print(f"total : -{total} états ({total / chaine[0][1]:.2%}) de 210 à 184")

chaîne de compression documentée par le dépôt (états) :
  nf-bb210             210 -> étape intermédiaire  201 : -9 état(s), 4.29% de la taille précédente
  étape intermédiaire  201 -> nf-bb192             192 : -9 état(s), 4.48% de la taille précédente
  nf-bb192             192 -> nf-bb185             185 : -7 état(s), 3.65% de la taille précédente
  nf-bb185             185 -> nf-bb184             184 : -1 état(s), 0.54% de la taille précédente
total : -26 états (12.38%) de 210 à 184


### Lecture de la chaîne

Le calcul ci-dessus montre que la compression est **modeste en pourcentage** : vingt-six états retirés sur deux cent dix, soit un peu plus de douze pour cent. Ce qui compte n'est pas l'ampleur, c'est ce que chaque état retiré **achète** : à chaque étape, la même équivalence d'arrêt doit rester un théorème, sur une table plus petite. Le dépôt documente les échecs intermédiaires, et c'est le point méthodologique du carnet : un contre-modèle à trois éléments pour `collapsed183` est une **information**, pas un déchet — il dit exactement ce qui casse quand on retire un état de trop.

La leçon est transposable : dans toute compression de machine, l'invariant qui doit survivre n'est pas la forme de la table, c'est le **théorème d'équivalence** qui la relie à la théorie. Les états décodeurs partagés sont un levier ; la vérification Lean est le juge.

In [5]:
# Discipline de l'empreinte : on hache ce qu'on possède, jamais ce qu'on cite.
formule_jouet = "D(x,y)=(J(Kx))y; ((D(x,y)=x)->(D(x,x)=x))->y=x"
empreinte = hashlib.sha256(formule_jouet.encode("utf-8")).hexdigest()

print("formule-jouet :", formule_jouet)
print("empreinte SHA-256 :", empreinte)
print("longueur :", len(empreinte), "caractères hexadécimaux")
print()
print("Empreinte publiée du paquet 184 : 7764cc84...8798 (citée du dépôt, révision 1297cce204).")
print("Elle n'est PAS re-vérifiable ici : la table NF184.tm n'est pas vendorée (aucune licence).")
print("Citée ne veut pas dire vérifiée -- c'est la règle de la section 0.")

formule-jouet : D(x,y)=(J(Kx))y; ((D(x,y)=x)->(D(x,x)=x))->y=x
empreinte SHA-256 : 8c83e4ed6e2eb9e05f81d9f29090af521bfc3af2da20924f9b2a10b409896719
longueur : 64 caractères hexadécimaux

Empreinte publiée du paquet 184 : 7764cc84...8798 (citée du dépôt, révision 1297cce204).
Elle n'est PAS re-vérifiable ici : la table NF184.tm n'est pas vendorée (aucune licence).
Citée ne veut pas dire vérifiée -- c'est la règle de la section 0.


### Ce que ce carnet ne peut pas faire

- **Il ne vérifie pas la table de 184 états.** Sans vendoring, l'empreinte publiée reste une citation ; le `SHA256SUMS` du dépôt source est l'instrument qui la contrôle, et il s'exécute là-bas.
- **Il ne prouve rien sur NF.** La cohérence de NF reste ouverte ; ce que les machines déplacent, c'est la **forme** de la question : « NF est-elle cohérente ? » devient « cette machine s'arrête-t-elle ? ». Le déplacement est utile parce qu'il rend l'énoncé fini et vérifiable, pas parce qu'il le résout.
- **Il ne fusionne pas les familles.** PA196 (carnet 08d, Greffe A) est une autre théorie, un autre dépôt, un autre statut. Toute prose qui mélange PA et NF rend les deux résultats inutilisables.

## 4. Les sept théories interprétables (pont vers formal_logic)

Le dossier `nf-bb192/interpretations` publie un index théorème par théorème (`SET-THEORY-PROOFS.md`) et une archive `nf-interp.zip` couvrant **sept** théories : NF, FunctionNF, CombNF, M, MAC, M+H et NFU + Infinity. C'est un objet d'une autre nature que les machines : une carte d'**interprétabilité** et d'**équiconsistance**, avec ses fichiers Lean originaux et leurs expositions (inventaire : SET-THEORY-PROOFS.md).

Le pont est direct vers la série `SymbolicAI/Lean` (formal_logic) : l'index cite la formulation exacte de H utilisée, et la question « quelles théories sont équiconsistantes ? » est le pendant sémantique de la question « quelles machines s'arrêtent ? ». Le carnet s'arrête ici volontairement : la consommation de cet index est un jalon distinct (Jalon 3 de [#27537](https://github.com/jsboige/CoursIA/issues/27537)), qui exigera de lire la copie archivée au gisement avant d'écrire — la règle bibliographique du dépôt.

## 5. Exercices

Trois exercices, stubs exécutables sans erreur volontaire (C.1) : le carnet s'exécute de bout en bout même non complétés.

### Exercice 1 — La cible du compte à rebours

Notre machine s'arrête sur la **clause vide**, c'est-à-dire sur une contradiction. Généralisez-la : une machine qui s'arrête sur une clause `cible` donnée, et seulement sur elle. La propriété à vérifier ensuite est la même, décalée d'un cran : `halte <=> cible appartient à la fermeture(kb)`.

*Indice* : la clause vide est le cas particulier `cible = frozenset()`. Le changement porte sur le test d'arrêt dans l'état `RESOUT`, pas sur l'énumération.

In [6]:
def halte_si_derive(kb: list[Clause], cible: Clause, max_pas: int = 200_000) -> tuple[bool | None, int]:
    # TODO etudiant : renvoyer (halte, pas) pour une machine qui s'arrête sur `cible`.
    # Indice : dupliquer CompteARebours en paramétrant le test d'arrêt.
    # Vérification attendue : comparer halte_si_derive(K3, frozenset({2})) à fermeture(K3).
    return None, 0


halte, nb_pas = halte_si_derive(K3, frozenset({2}))
print("Exercice à compléter" if halte is None else f"K3 dérive {{2}} : halte={halte} en {nb_pas} pas")

Exercice à compléter


### Exercice 2 — Rétrécir sans perdre l'équivalence

La chaîne réelle descend de 210 à 184 états. Écrivez le profil de réduction : pour une suite d'entiers, le pourcentage retiré à chaque étape et le total. Puis répondez en une phrase, dans une cellule markdown : quel invariant doit survivre à **chaque** compression, et pourquoi un contre-modèle conservé (le `collapsed183` à trois éléments) vaut-il un résultat ?

*Indice* : `profil_reduction([210, 201, 192, 185, 184])` doit renvoyer quatre valeurs, et la somme des retraits doit redonner vingt-six états.

In [7]:
def profil_reduction(etats: list[int]) -> list[float] | None:
    # TODO etudiant : renvoyer les réductions successives, en pourcentage du précédent.
    # Indice : quatre valeurs pour cinq états ; le total doit être cohérent.
    return None


profil = profil_reduction([210, 201, 192, 185, 184])
print("Exercice à compléter" if profil is None else f"profil : {profil}")

Exercice à compléter


### Exercice 3 — Classer un résultat avant de le citer

On vous donne trois affirmations. Classez chacune dans l'un des trois statuts de la section 0 : `formalise-audite`, `calcul-reproductible` ou `borne-conditionnelle`.

a) « La machine à 185 états s'arrête si et seulement si NF est réfutable, et l'audit ne liste que les axiomes standard. »
b) « Un simulateur indépendant a exécuté 142 608 essais sur la table à 184 états, sans divergence. »
c) « Si NF est cohérente, alors NFU + Infinity est cohérente. »

*Indice* : demandez-vous, pour chaque énoncé, **qui** peut le vérifier et **avec quel artefact** — c'est la question qui distingue un statut d'un autre.

In [8]:
def classer(enonce: str) -> str | None:
    # TODO etudiant : renvoyer "formalise-audite", "calcul-reproductible" ou "borne-conditionnelle".
    return None


enonces = [
    "La machine à 185 états s'arrête si et seulement si NF est réfutable (audit d'axiomes publié).",
    "Un simulateur indépendant a exécuté 142 608 essais sur la table à 184 états.",
    "Si NF est cohérente, alors NFU + Infinity est cohérente.",
]
for enonce in enonces:
    verdict = classer(enonce)
    print("Exercice à compléter" if verdict is None else f"{verdict} : {enonce[:60]}...")

Exercice à compléter
Exercice à compléter
Exercice à compléter


## 6. Conclusion et ponts

Le compte à rebours est une **traduction** : il transforme une question sur une théorie (« NF est-elle cohérente ? ») en une question sur une machine finie (« s'arrête-t-elle ? »). Trois exigences en découlent, et le carnet les a tenues séparément :

1. **L'équivalence doit être un théorème.** C'est le statut des machines 185 et 192, auditées. Le jouet de la section 2 n'a pas ce statut : il illustre le câblage, il ne prouve rien.
2. **La table doit être reproductible.** C'est le statut du paquet 184 : empreinte, simulateur indépendant, points de contrôle. Citer une empreinte n'est pas la vérifier.
3. **Une borne se cite avec sa condition.** C'est le statut de C₀-198 : Con(NF) ⇒ non-arrêt ⇒ Con(NFU + Infinity), condition comprise.

Les ponts naturels sont la série `SymbolicAI/Lean` (l'index d'interprétabilité des sept théories, Jalon 3) et le carnet 07b (le castor affairé comme fonction de croissance, dont ces machines calculent des valeurs inférieures). Le Jalon 2 de [#27537](https://github.com/jsboige/CoursIA/issues/27537) — un lac consommateur `nf_bb_lean` épinglé sur NF185 — viendra quand la fenêtre de build bornée sera libre ; ce carnet prépare sa prose sans emprunter sa dette.

## 7. Références

**Source primaire — dépôt `wlad-svennik/notes`** (lu firsthand le 20/11/2026, dernier push 11:19Z ; révisions épinglées citées en section 3). Aucune licence détectée dans le dépôt ; demande de clarification ouverte : `wlad-svennik/notes#1`. Aucun contenu recopié.

**Copies archivées au gisement** (bibliography-hygiene, identités vérifiées page 1), rayon `G:\Mon Drive\MyIA\IA\Bibliographie IA\Automata\Complexity Theory and Hardness\` :

- `2026 - Svennik (wlad) - 192-state PA bound for non-regularity (github wlad-svennik-notes, sans licence).pdf`
- `2026 - Svennik (wlad) - 192-state countdown machine and NF (github wlad-svennik-notes, sans licence).pdf`
- `2026 - Svennik (wlad) - tangled type theory and Con NF (github wlad-svennik-notes, sans licence).pdf`
- `2026 - Svennik (wlad) - 201-state Zermelo (github wlad-svennik-notes, sans licence).pdf`

**Références de fond, non consommées dans ce carnet** : Quine, *New Foundations for Mathematical Logic* (1937) — compréhension stratifiée ; Jensen, résultats de cohérence de NFU ; le carnet [Complexity-07b](Complexity-07b-LeTourDesGrandsNombres-Python.ipynb) pour le castor affairé et Tibor Radó (1962) ; [Complexity-08d](https://github.com/jsboige/CoursIA/pull/27483) (Greffe A) pour PA196, qui n'est **pas** fusionné avec ce carnet.

*Greffe D de [#27537](https://github.com/jsboige/CoursIA/issues/27537), EPIC [#20607](https://github.com/jsboige/CoursIA/issues/20607).*